# 02 — Feature Engineering
Builds the full feature catalog described in `docs/feature_catalog.md` (Deliverable D2.1.3, Section A5).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.data.synthetic_generator import SupplyChainDataGenerator
gen = SupplyChainDataGenerator(seed=42)
nodes, edges = gen.generate_graph()
ts = gen.generate_time_series(n_days=365)
shipments = gen.generate_shipments(n=2000)
lc = gen.generate_trade_finance_transactions(n=2000, nodes=nodes)

## Entity (graph) features — Section A5.1

In [2]:
from src.features.graph_features import build_entity_features, normalise_features, ENTITY_FEATURE_COLUMNS
entity_feats = build_entity_features(nodes)
print(f'{len(ENTITY_FEATURE_COLUMNS)} entity features')
entity_feats[ENTITY_FEATURE_COLUMNS].describe().T

21 entity features


,count,mean,std,min,25%,50%,75%,max
current_ratio,217.0,1.457234,0.484890,0.400000,1.099191,1.480535,1.711234,3.089932
debt_to_equity,217.0,0.822096,0.557127,0.050000,0.431647,0.720756,1.091568,4.184060
ebitda_margin,217.0,0.131379,0.060489,-0.047872,0.089381,0.134419,0.172864,0.282100
interest_coverage,217.0,4.627930,2.698923,0.285408,2.796794,3.865645,6.204238,14.333274
working_capital_ratio,217.0,0.072862,0.024244,0.020000,0.054960,0.074027,0.085562,0.154497
otif_rate,217.0,0.894030,0.068839,0.684888,0.847930,0.900789,0.950827,0.995000
lead_time_mean,217.0,14.044663,5.037818,2.000000,10.249991,13.646598,17.257703,28.287192
lead_time_std,217.0,2.939852,2.046753,0.300000,1.403331,2.593350,3.852421,14.293752
inventory_turnover,217.0,6.480254,2.040783,1.268281,5.075989,6.423706,7.908470,11.838669
supplier_concentration_hhi,217.0,0.257519,0.148189,0.050000,0.150896,0.235495,0.331472,0.778549


## Temporal features — Section A5.2

In [3]:
from src.features.temporal_features import build_temporal_features
port0 = ts[ts.port == ts.port.iloc[0]]
temporal_feats = build_temporal_features(port0, value_col='throughput_teu')
new_cols = [c for c in temporal_feats.columns if c not in port0.columns]
print(f'{len(new_cols)} engineered temporal features')
temporal_feats[new_cols].head()

37 engineered temporal features


,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_30,roll_std_30,roll_mean_90,roll_std_90,roll_min_7,roll_max_7,...,lag_1,lag_7,lag_14,lag_30,fourier_sin_365,fourier_cos_365,fourier_sin_182,fourier_cos_182,fourier_sin_91,fourier_cos_91
0,74359.00,0.000000,74359.00,0.000000,74359.00,0.000000,74359.00,0.000000,74359.0,74359.0,...,74359.0,74359.0,74359.0,74359.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
1,76522.50,3059.651042,76522.50,3059.651042,76522.50,3059.651042,76522.50,3059.651042,74359.0,78686.0,...,74359.0,74359.0,74359.0,74359.0,0.017202,0.999852,0.034398,0.999408,0.068755,0.997634
2,77679.00,2948.424834,77679.00,2948.424834,77679.00,2948.424834,77679.00,2948.424834,74359.0,79992.0,...,78686.0,74359.0,74359.0,74359.0,0.034398,0.999408,0.068755,0.997634,0.137185,0.990545
3,73702.75,8308.894566,73702.75,8308.894566,73702.75,8308.894566,73702.75,8308.894566,61774.0,79992.0,...,79992.0,74359.0,74359.0,74359.0,0.051584,0.998669,0.103031,0.994678,0.204966,0.978769
4,71072.40,9293.653872,71072.40,9293.653872,71072.40,9293.653872,71072.40,9293.653872,60551.0,79992.0,...,61774.0,74359.0,74359.0,74359.0,0.068755,0.997634,0.137185,0.990545,0.271777,0.962360


## Trade finance transaction + cross-domain fusion features — Sections A5.3, A5.4

In [4]:
from src.features.financial_features import (
    build_lc_transaction_features, LC_TRANSACTION_FEATURE_COLUMNS,
    supply_chain_adjusted_pd, working_capital_velocity_index, trade_route_financial_stress_index)
lc_feats = build_lc_transaction_features(lc)
print(f'{len(LC_TRANSACTION_FEATURE_COLUMNS)} LC transaction features')
lc_feats[LC_TRANSACTION_FEATURE_COLUMNS].describe().T

15 LC transaction features


,count,mean,std,min,25%,50%,75%,max
lc_amount_log,2000.0,13.153077,1.145631,9.366232,12.384137,13.157248,13.945938,16.755561
tenor_days,2000.0,95.190000,51.610035,30.000000,60.000000,90.000000,120.000000,180.000000
commodity_risk_num,2000.0,0.866000,0.727536,0.000000,0.000000,1.000000,1.000000,2.000000
trade_route_risk_score,2000.0,0.337415,0.176743,0.013000,0.197000,0.320000,0.460000,0.905000
applicant_leverage,2000.0,0.882061,0.550439,0.050000,0.464000,0.814500,1.190000,2.572000
applicant_current_ratio,2000.0,1.420152,0.456257,0.532000,1.074000,1.461000,1.711000,3.090000
beneficiary_otif_rate,2000.0,0.895218,0.069399,0.723900,0.847300,0.903850,0.947850,0.995000
historical_discrepancy_rate_applicant,2000.0,0.283922,0.157732,0.002300,0.159850,0.260000,0.388275,0.809700
historical_discrepancy_rate_beneficiary,2000.0,0.198338,0.137848,0.001100,0.093250,0.171050,0.271700,0.757200
port_congestion_origin,2000.0,2.037785,0.891808,0.100000,1.450000,2.035000,2.660000,5.000000


In [5]:
# Reproduce the brief's SC-PD worked example exactly (Section A5.4)
sc_pd = supply_chain_adjusted_pd(0.025, otif_actual=0.85, inv_turnover_actual=4.8, alternative_supplier_count=1)
print(f'SC-PD = {sc_pd:.4f} (brief worked example: 0.0333)')

SC-PD = 0.0333 (brief worked example: 0.0333)
